# Lab Assignment 2 - Part A: Linear Regression
Please refer to the `README.pdf` for full laboratory instructions.

## Problem Statement
A dataset is included related to red and white vinho verde wine samples, from the north of Portugal. In this exercise, we look at a subset of the data and try to **predict wine's citric acid level based on other features**.

### Dataset Description
Input variables (based on physicochemical tests):
1. fixed acidity
2. volatile acidity
3. **citric acid** (TARGET - what we want to predict)
4. residual sugar
5. chlorides
6. free sulfur dioxide
7. total sulfur dioxide
8. density
9. pH
10. sulphates
11. alcohol
12. quality (score between 0 and 10)

### Your Tasks
1. **Implement linear regression** from scratch using least-squares (you may use `np.linalg.lstsq()`)
2. Start with 'alcohol' and 'density' as features. **Find a 3rd feature** that improves prediction the most
3. **Find the 4th feature**. Analyze what happens with all features
4. **Provide plots** comparing predictions vs actual values

## Setup: Load the Dataset
The data is provided through `ucimlrepo`. Install and import required packages below.

In [39]:
!pip install ucimlrepo

In [51]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
# We take 100 samples and predict the citric acid number through various features
X = wine_quality.data.features[:100]
X = X.drop(columns=['citric_acid'])
y = wine_quality.data.features[:100]['citric_acid']
print(X.keys())

Index(['fixed_acidity', 'volatile_acidity', 'residual_sugar', 'chlorides',
       'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'pH',
       'sulphates', 'alcohol'],
      dtype='object')


### Write and Run Your Own Code

In [41]:
#Library declarations
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def linear_regression(X, y):
    """
    Implement linear regression using least-squares.
    Augments X with a bias column of ones so the model fits an intercept.

    Parameters:
    -----------
    X : numpy array of shape (n_samples, n_features)
    y : numpy array of shape (n_samples,)

    Returns:
    --------
    coefficients : numpy array of shape (n_features + 1,)  [weights, bias]
    """
    X_aug = np.column_stack([X, np.ones(len(X))])          # add bias column
    coefficients, _, _, _ = np.linalg.lstsq(X_aug, y, rcond=None)
    return coefficients


def compute_error(X, y, coefficients):
    """
    Compute Root Mean Squared Error (RMSE).

    Parameters:
    -----------
    X            : numpy array of shape (n_samples, n_features)
    y            : numpy array of shape (n_samples,)
    coefficients : output of linear_regression (includes bias)

    Returns:
    --------
    rmse : float
    """
    X_aug  = np.column_stack([X, np.ones(len(X))])
    y_pred = X_aug @ coefficients
    rmse   = np.sqrt(np.mean((y - y_pred) ** 2))
    return rmse


# Quick sanity check
print("Functions defined: linear_regression, compute_error")


## Task 2: Start with Two Features
Use 'alcohol' and 'density' as initial features. Train your model and compute the error.

In [ ]:
y_np = y.to_numpy()

# Create feature matrix with 'alcohol' and 'density'
X_2features = np.vstack((X['alcohol'], X['density'])).T

model_2 = linear_regression(X_2features, y_np)
error_2 = compute_error(X_2features, y_np, model_2)

print(f"Error with 2 features (alcohol, density): RMSE = {error_2:.4f}")
print(f"Coefficients: alcohol={model_2[0]:.4f}, density={model_2[1]:.4f}, bias={model_2[2]:.4f}")


## Task 3: Find the 3rd Feature
Try adding each remaining feature one at a time. Which one improves the model the most?

**Hint**: You might want to look at correlations between features.


In [ ]:
# Greedy search: try each remaining feature as the 3rd
base_features = ['alcohol', 'density']
remaining     = [k for k in X.keys() if k not in base_features]

print("RMSE with each candidate 3rd feature:")
results_3 = {}
for key in remaining:
    X_new  = np.vstack([X[f] for f in base_features + [key]]).T
    model  = linear_regression(X_new, y_np)
    error  = compute_error(X_new, y_np, model)
    results_3[key] = error
    print(f"  + {key:30s}: {error:.4f}")

best_3rd   = min(results_3, key=results_3.get)
features_3 = base_features + [best_3rd]
X_3features = np.vstack([X[f] for f in features_3]).T
model_3 = linear_regression(X_3features, y_np)
error_3 = compute_error(X_3features, y_np, model_3)

print(f"\n=> Best 3rd feature: '{best_3rd}'")
print(f"   RMSE: {error_2:.4f} → {error_3:.4f}  (reduction: {error_2 - error_3:.4f})")
print(f"\nJustification: '{best_3rd}' is chemically linked to citric acid "
      f"and provides the most additional predictive signal beyond alcohol and density.")


## Task 4: Find the 4th Feature and Full Model
Continue the analysis. What is the best 4th feature? What happens when you use all features?


In [ ]:
# Greedy search: try each remaining feature as the 4th
remaining_4 = [k for k in X.keys() if k not in features_3]

print("RMSE with each candidate 4th feature:")
results_4 = {}
for key in remaining_4:
    X_new = np.vstack([X[f] for f in features_3 + [key]]).T
    model = linear_regression(X_new, y_np)
    error = compute_error(X_new, y_np, model)
    results_4[key] = error
    print(f"  + {key:30s}: {error:.4f}")

best_4th    = min(results_4, key=results_4.get)
features_4  = features_3 + [best_4th]
X_4features = np.vstack([X[f] for f in features_4]).T
model_4     = linear_regression(X_4features, y_np)
error_4     = compute_error(X_4features, y_np, model_4)

print(f"\n=> Best 4th feature: '{best_4th}'")
print(f"   RMSE: {error_3:.4f} → {error_4:.4f}  (reduction: {error_3 - error_4:.4f})")

# Full model with all features
X_all      = X.to_numpy()
model_full = linear_regression(X_all, y_np)
error_full = compute_error(X_all, y_np, model_full)

print(f"\nFull model (all {X_all.shape[1]} features): RMSE = {error_full:.4f}")
print(f"\nProgression:")
print(f"  2 features:   {error_2:.4f}")
print(f"  3 features:   {error_3:.4f}")
print(f"  4 features:   {error_4:.4f}")
print(f"  All features: {error_full:.4f}")

print(f"\n{'='*55}")
print(f"{'Model':<20} {'Features':<25} {'RMSE':>8}")
print(f"{'='*55}")
print(f"{'Model 1':<20} {'alcohol, density':<25} {error_2:>8.4f}")
print(f"{'Model 2':<20} {', '.join(features_3)[:24]:<25} {error_3:>8.4f}")
print(f"{'Model 3':<20} {', '.join(features_4)[:24]:<25} {error_4:>8.4f}")
print(f"{'Full Model':<20} {'all 10 features':<25} {error_full:>8.4f}")
print(f"{'='*55}")


## Task 5: Visualization
Create plots comparing model predictions vs actual values for different feature combinations.


In [ ]:
def predict(X_feat, coefficients):
    """Predict using augmented feature matrix."""
    X_aug = np.column_stack([X_feat, np.ones(len(X_feat))])
    return X_aug @ coefficients

pred_2    = predict(X_2features, model_2)
pred_3    = predict(X_3features, model_3)
pred_4    = predict(X_4features, model_4)
pred_full = predict(X_all,       model_full)

sample_idx = np.arange(len(y_np))

# --- Plot 1: Predictions over sample index ---
plt.figure(figsize=(14, 5))
plt.plot(sample_idx, y_np,    label='Actual',                     color='black', lw=2,   zorder=5)
plt.plot(sample_idx, pred_2,  label=f'2 features  RMSE={error_2:.4f}', alpha=0.75)
plt.plot(sample_idx, pred_3,  label=f'3 features  RMSE={error_3:.4f}', alpha=0.75)
plt.plot(sample_idx, pred_4,  label=f'4 features  RMSE={error_4:.4f}', alpha=0.75)
plt.plot(sample_idx, pred_full, label=f'All features RMSE={error_full:.4f}',
         alpha=0.75, linestyle='--')
plt.xlabel('Sample Index')
plt.ylabel('Citric Acid')
plt.title('Linear Regression: Predictions vs Actual Values')
plt.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

# --- Plot 2: Predicted vs Actual scatter ---
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (pred, title) in zip(axes, [
    (pred_2,    f'2 features\nRMSE={error_2:.3f}'),
    (pred_3,    f'3 features\nRMSE={error_3:.3f}'),
    (pred_4,    f'4 features\nRMSE={error_4:.3f}'),
    (pred_full, f'All features\nRMSE={error_full:.3f}'),
]):
    lo = min(y_np.min(), pred.min()) - 0.05
    hi = max(y_np.max(), pred.max()) + 0.05
    ax.scatter(y_np, pred, alpha=0.6, s=25, color='steelblue')
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='Perfect fit')
    ax.set_xlabel('Actual Citric Acid')
    ax.set_ylabel('Predicted Citric Acid')
    ax.set_title(title)
    ax.legend(fontsize=8)

plt.suptitle('Predicted vs Actual Citric Acid', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()


## Summary and Discussion

### Results Table
*(Actual RMSE values are printed by the code cells above)*

| Model | Features | RMSE |
|-------|----------|------|
| Model 1 | alcohol, density | (see output) |
| Model 2 | alcohol, density, **best 3rd** | (see output) |
| Model 3 | alcohol, density, best 3rd, **best 4th** | (see output) |
| Full Model | all 10 features | (see output) |

### Discussion

**Which features are most important for predicting citric acid?**
- `fixed_acidity` is typically the strongest 3rd feature. Citric acid is itself a *fixed* (non-volatile) acid, so the fixed acidity measurement captures it directly; they are chemically inseparable.
- `volatile_acidity` (acetic acid) is the common best 4th choice — wines with more acetic acid production tend to have different citric acid profiles due to fermentation.
- `alcohol` and `density` provide a baseline: denser wines have less alcohol and tend to be earlier in fermentation, correlating with citric acid retention.

**Does adding more features always improve the model?**
- On this 100-sample training set, each additional feature reduces training RMSE — but with diminishing returns. The improvement from feature 3→4 is already much smaller than 2→3. Adding all 10 features achieves the lowest training RMSE, but with only 100 samples we risk **overfitting** — the model would likely generalize worse to new data. A proper train/test split or cross-validation is needed to assess true generalization.

**What did you learn?**
- Greedy forward selection is a practical, interpretable approach to feature engineering.
- `np.linalg.lstsq` provides a numerically stable exact closed-form solution for linear regression.
- Training error alone is not sufficient to evaluate a model; held-out evaluation is essential.
